[Python, Visually](https://johnfisher-ai.github.io/Python-Visual-Guides/) &nbsp;&rsaquo;&nbsp; [sqlite3, Deep Dive](https://johnfisher-ai.github.io/Python-Visual-Guides/sqlite3-deep-dive.html)

# A Searchable Archive


## What you will be able to do

Build a small application on one SQLite file, from a folder of text files to a backup: a schema
created and changed by numbered migrations, a loader that can run every night without making a
second copy of a document or losing what people added, a full-text search that ranks what it finds,
lookups that use an index, searches that run while the loader writes, and a checked backup to
download. Name the notebook of this guide that every part of it comes from.


## The idea

### The problem

The technicians at the four stations keep a logbook, and the office keeps it as text files: a folder
for every station, and a file for every month of 2025, 48 files in all. Questions about it come in
every week. Which months mention a battery, and which of them most? Which of Svalbard's summer months
has somebody already read? Opening files one at a time answers the first question slowly and the
second not at all, and a text search finds the lines but cannot rank them, and knows nothing people
have noted about a file.

A database answers both, if it is built for the way the folder is used. The folder changes: a
technician corrects a file, and a new month arrives, so the loader runs every night, and a second run
must neither fail on the documents already there nor copy them. The questions change too, so the
schema will change after people have started marking what they have read, and last month's archive
has to be brought up to date without losing a mark. People search while the loader writes, so
neither can lock out the other. And the archive is one file, so it needs a backup that is really one.

### What a searchable archive is

> A **searchable archive**, in this notebook, is one SQLite file that holds a collection of
> documents: a `documents` table with a row for every file, keyed by its **path**, beside what people
> have added, such as a mark for the documents someone has **reviewed**; an FTS5 index of the
> documents' words, which **triggers** keep in step with the table; and the **schema version** in
> `PRAGMA user_version`. Its **loader** is **idempotent**: running it again on the same folder
> changes nothing, and running it after a file has changed rewrites that document's text and nothing
> else. Every change to the archive's shape is a numbered **migration**, run once for every archive
> that has not had it.

### Why it works that way

- **The path is the key.** A file keeps its path when its text changes, so `UNIQUE` on `path` with
  `ON CONFLICT (path) DO UPDATE` turns a second load of a file into an update of the same row, which
  keeps its `id` and every column the loader does not write.
- **`RETURNING` reports what changed.** With a `WHERE` on the update, a document whose text is the
  same is not written at all and returns no row, so the rows returned are the documents that changed.
- **The index reads the table.** An external content index stores only the words, and the triggers
  on `documents` keep it in step, so the loader writes to one table and search is never behind.
- **The version is in the file.** `PRAGMA user_version` travels with the archive, so code that opens
  an archive made by older code runs exactly the migrations that archive lacks.
- **WAL lets a search and a load overlap.** A search reads the last commit while the loader's
  transaction is open, and neither waits for the other.
- **One file, so one backup.** The documents, their index and the version are all in the file, so
  a copy made with `VACUUM INTO` searches exactly as the archive does.

### Where this shows up

calibre, the e-book manager, keeps the details of its library in a SQLite file, and searches the
full text of its books with FTS5. Datasette publishes SQLite files on the web, and puts a search box
on any table that has a full-text index. In this library, the **SQLAlchemy, Deep Dive** guide builds
applications like this one with models and sessions, and changes their schemas with Alembic, which
numbers migrations as this notebook does by hand. The libraries in the **LlamaIndex, Deep Dive** and
**Haystack, Deep Dive** guides search collections of documents too, and offer the BM25 ranking this
notebook uses beside search by meaning.

### What this notebook covers

- The folder of monthly logbook files
- Version 1 of the archive, written as a migration
- A loader that can run every night, and says what changed
- Searching the archive, best match first
- A lookup by station and month that uses an index
- Version 2, a column for reviews, added to an archive in use
- Searching while the loader writes
- A checked backup, searched and downloaded
- When to use `INSERT`, `DO NOTHING`, `DO UPDATE` or `INSERT OR REPLACE` in a loader
- Three nights of an archive made by last month's code
- Seven errors: a loader that only inserts, a file that is not UTF-8, a report on an archive at
  version 1, a migration run twice, a search that locks out the loader, `VACUUM` inside the load's
  transaction, and `INSERT OR REPLACE` in the loader

### A first look

Before any of the detail, here is the whole idea in a few lines. There is nothing to run yet: read
it, and read the output underneath it. Everything from Setup onward is where you start running
things, and the rest of the notebook takes this apart piece by piece.

```python
import sqlite3
from pathlib import Path

folder = Path("logbook")
folder.mkdir()
(folder / "bergen.txt").write_text("Battery replaced after a low voltage warning.", encoding="utf-8")
(folder / "oslo.txt").write_text("Snow cleared from the rain gauge.", encoding="utf-8")

archive = sqlite3.connect(":memory:", autocommit=True)
archive.execute("CREATE TABLE documents (path TEXT UNIQUE, body TEXT)")
for night in (1, 2):
    for path in sorted(folder.glob("*.txt")):
        archive.execute("""INSERT INTO documents VALUES (?, ?)
                           ON CONFLICT (path) DO UPDATE SET body = excluded.body""",
                        (path.name, path.read_text(encoding="utf-8")))
    count = archive.execute("SELECT COUNT(*) FROM documents").fetchone()[0]
    print(f"after night {night}: {count} documents")

archive.execute("CREATE VIRTUAL TABLE search USING fts5(path, body)")
archive.execute("INSERT INTO search SELECT path, body FROM documents")
print(archive.execute("SELECT path FROM search WHERE search MATCH 'gauge'").fetchall())
archive.close()
```

```
after night 1: 2 documents
after night 2: 2 documents
[('oslo.txt',)]
```

The second night loaded the same two files and still left two documents, since the upsert turned a
path already there into an update. The full-text index then found the document about the rain gauge
by one of its words. This notebook builds the same thing properly: an index that keeps itself in
step, a version, a search while the loader writes, and a backup.


## Setup

Six imports, and the logbook the **Full-Text Search** notebook generated, written out as a folder of
text files, with `as_words` from the same notebook.

- `sqlite3` holds the archive
- `math`, `datetime` and `timedelta` make the year of readings the notes are written from
- `contextmanager`, from `contextlib`, turns a function into the `with` block for the archive's
  transactions
- `Path` names the folder, its files and the archive
- `shutil` removes the scratch folder at the start and at the end

`write_logbook` writes a file for every station and month, `<station>/<month>.txt`, with a heading
and a line for every day's note, and returns how many files it wrote.


In [1]:
import math
import shutil
import sqlite3
from contextlib import contextmanager
from datetime import datetime, timedelta
from pathlib import Path

SCRATCH = Path("scratch")
shutil.rmtree(SCRATCH, ignore_errors=True)
SCRATCH.mkdir()
FOLDER = SCRATCH / "logbook"
ARCHIVE = SCRATCH / "archive.db"
STATIONS = {"Bergen": 8.0, "Oslo": 6.5, "Svalbard": -4.5, "Tromso": 3.5}   # each station's mean for the year
EVENTS = [
    "Heater on the sensor mast checked and working.",
    "Battery replaced after a low voltage warning.",
    "Snow cleared from the rain gauge.",
    "Sensor recalibrated against the reference thermometer.",
    "Ice on the anemometer, so the wind readings for the morning are unreliable.",
    "Annual service of the station completed.",
    "Fence repaired after a storm.",
    "Data logger restarted after a power cut, and no readings were lost.",
    "Heaters on the mast replaced.",
    "Visited twice to check the heater.",
]
ON_WARM_DAYS = {EVENTS[2]: "Grass cut around the rain gauge.", EVENTS[4]: "Anemometer bearings greased."}


def year_of_readings():
    """Every hour of 2025 at the four stations, as Why sqlite3 made them, with Svalbard silent on 2 March."""
    for n in range(365 * 24):
        hour = datetime(2025, 1, 1) + timedelta(hours=n)
        season = -math.cos(2 * math.pi * (n - 400) / (365 * 24))
        day = -math.cos(2 * math.pi * (hour.hour - 3) / 24)
        for i, (station, mean) in enumerate(STATIONS.items()):
            if station == "Svalbard" and hour.strftime("%Y-%m-%d") == "2025-03-02":
                celsius = None
            else:
                wobble = ((n * 37 + i * 101) % 17 - 8) / 10
                celsius = round(mean + 9 * season + 3 * day + wobble, 1) + 0.0
            yield station, hour.strftime("%Y-%m-%dT%H:%M"), celsius


def logbook():
    """A technician's note for every station and day of 2025, written from that day's readings."""
    days = {}
    for station, hour, celsius in year_of_readings():
        days.setdefault((hour[:10], station), []).append(celsius)
    for (day, station), temperatures in sorted(days.items()):
        known = [celsius for celsius in temperatures if celsius is not None]
        if not known:
            yield station, day, "Data logger failed overnight, and there are no readings for the whole day."
            continue
        low, high = min(known), max(known)
        if high < 0:
            weather = f"Frost all day, between {low} and {high} degrees."
        elif low < 0:
            weather = f"Night frost down to {low} degrees, and a thaw to {high} by the afternoon."
        else:
            weather = f"Above freezing all day, between {low} and {high} degrees."
        day_of_year = datetime.strptime(day, "%Y-%m-%d").timetuple().tm_yday
        number = (day_of_year * 37 + list(STATIONS).index(station) * 101) % 23
        event = EVENTS[number] if number < len(EVENTS) else ""
        if low >= 0:
            event = ON_WARM_DAYS.get(event, event)
        yield station, day, f"{weather} {event}".strip()


def write_logbook(folder):
    """Write the logbook as a file for every station and month, folder/<station>/<month>.txt, and return how many."""
    months = {}
    for station, day, note in logbook():
        months.setdefault((station, day[:7]), []).append(f"{day}: {note}")
    for (station, month), lines in months.items():
        path = folder / station / f"{month}.txt"
        path.parent.mkdir(parents=True, exist_ok=True)
        heading = f"{station}, {datetime.strptime(month, '%Y-%m'):%B %Y}"
        path.write_text(heading + "\n\n" + "\n".join(lines) + "\n", encoding="utf-8")
    return len(months)


def as_words(text):
    """Text typed into a search box, as an FTS5 query that needs every word and treats none as an operator."""
    return " ".join('"' + word.replace('"', '""') + '"' for word in text.split())


print("wrote", write_logbook(FOLDER), "files to", FOLDER)


wrote 48 files to scratch/logbook


## Worked examples

### The folder

A file starts with a heading, and has a line for every day:


In [2]:
paths = sorted(FOLDER.glob("*/*.txt"))
print(len(paths), "files, from", paths[0].relative_to(FOLDER).as_posix(), "to", paths[-1].relative_to(FOLDER).as_posix())
print()
for line in (FOLDER / "Svalbard" / "2025-03.txt").read_text(encoding="utf-8").splitlines()[:5]:
    print(line)


48 files, from Bergen/2025-01.txt to Tromso/2025-12.txt

Svalbard, March 2025

2025-03-01: Frost all day, between -14.7 and -8.0 degrees. Data logger restarted after a power cut, and no readings were lost.
2025-03-02: Data logger failed overnight, and there are no readings for the whole day.
2025-03-03: Frost all day, between -14.2 and -7.2 degrees.


March at Svalbard includes the day its data logger failed. A station's name is the name of its
folder, and a month is the name of its file, which is all the loader needs to know about a document
besides its text.

### Version 1 of the archive

The archive's schema is written as a migration, the first in a list, as the **Changing a Schema**
notebook numbered them, so that a new file and an old archive are brought up to date by the same
code. `transaction` runs a `with` block inside `BEGIN IMMEDIATE` and `COMMIT`, or `ROLLBACK` if the
block raises, on a connection opened with `autocommit=True`, where the SQL writes its own
transactions. `open_archive` opens every connection to the archive the same way:


In [3]:
@contextmanager
def transaction(conn):
    """Run a with block inside BEGIN IMMEDIATE and COMMIT, or ROLLBACK if anything in it raises."""
    conn.execute("BEGIN IMMEDIATE")
    try:
        yield
        conn.execute("COMMIT")
    except BaseException:
        conn.execute("ROLLBACK")
        raise


VERSION_1 = [
    """
    CREATE TABLE documents (
        id INTEGER PRIMARY KEY,
        path TEXT NOT NULL UNIQUE,
        station TEXT NOT NULL,
        month TEXT NOT NULL,
        body TEXT NOT NULL
    ) STRICT
    """,
    "CREATE INDEX documents_by_station_month ON documents (station, month)",
    "CREATE VIRTUAL TABLE documents_index USING fts5(body, content = 'documents', content_rowid = 'id')",
    """
    CREATE TRIGGER documents_after_insert AFTER INSERT ON documents BEGIN
        INSERT INTO documents_index (rowid, body) VALUES (new.id, new.body);
    END
    """,
    """
    CREATE TRIGGER documents_after_delete AFTER DELETE ON documents BEGIN
        INSERT INTO documents_index (documents_index, rowid, body) VALUES ('delete', old.id, old.body);
    END
    """,
    """
    CREATE TRIGGER documents_after_update AFTER UPDATE OF body ON documents BEGIN
        INSERT INTO documents_index (documents_index, rowid, body) VALUES ('delete', old.id, old.body);
        INSERT INTO documents_index (rowid, body) VALUES (new.id, new.body);
    END
    """,
]


def create_archive(conn):
    """Version 1: the documents, an index for lookups by station and month, and a full-text index kept in step."""
    for statement in VERSION_1:
        conn.execute(statement)


def migrate(conn, migrations):
    """Run every migration the archive has not had, each in a transaction with its number, and return the numbers."""
    version = conn.execute("PRAGMA user_version").fetchone()[0]
    applied = []
    for number in range(version + 1, len(migrations) + 1):
        with transaction(conn):
            migrations[number - 1](conn)
            conn.execute(f"PRAGMA user_version = {number}")
        applied.append(number)
    return applied


MIGRATIONS = [create_archive]


def open_archive(path):
    """A connection to the archive at path, in WAL mode, with rows by column name, brought up to date."""
    conn = sqlite3.connect(path, autocommit=True)
    conn.row_factory = sqlite3.Row
    conn.execute("PRAGMA journal_mode = WAL")
    migrate(conn, MIGRATIONS)
    return conn


conn = open_archive(ARCHIVE)
print("version:", conn.execute("PRAGMA user_version").fetchone()[0])
for row in conn.execute("SELECT type, name FROM sqlite_schema WHERE name NOT LIKE 'documents_index_%' ORDER BY rowid"):
    print(f"    {row['type']:<8} {row['name']}")


version: 1
    table    documents
    index    sqlite_autoindex_documents_1
    index    documents_by_station_month
    table    documents_index
    trigger  documents_after_insert
    trigger  documents_after_delete
    trigger  documents_after_update


Opening a new file ran the one migration there is, so the archive is at version 1: the `documents`
table, the index SQLite made for `UNIQUE` on `path`, the index for lookups by station and month, the
full-text index, and its three triggers. The list leaves out the shadow tables FTS5 keeps its index
in. The update trigger is `AFTER UPDATE OF body`, so it runs only when a document's text changes, and
a change to any other column leaves the index alone. The statements run one at a time inside the
migration's transaction, so a failure in any of them leaves an empty file at version 0.

### A loader that can run every night

`load` reads every file and upserts it by its path, all in one transaction, and returns the paths
whose documents changed. Here it runs twice, and then again after a technician has added a line to
Oslo's February:


In [4]:
UPSERT_DOCUMENT = """
    INSERT INTO documents (path, station, month, body) VALUES (?, ?, ?, ?)
    ON CONFLICT (path) DO UPDATE SET body = excluded.body WHERE documents.body IS NOT excluded.body
    RETURNING id
"""


def load(conn, folder):
    """Load every text file in folder's station folders, in one transaction, and return the paths whose text changed."""
    changed = []
    with transaction(conn):
        for path in sorted(folder.glob("*/*.txt")):
            name = path.relative_to(folder).as_posix()
            text = path.read_text(encoding="utf-8")
            if conn.execute(UPSERT_DOCUMENT, (name, path.parent.name, path.stem, text)).fetchall():
                changed.append(name)
    return changed


print("first load, changed:", len(load(conn, FOLDER)), "documents")
print("second load, changed:", load(conn, FOLDER))

with (FOLDER / "Oslo" / "2025-02.txt").open("a", encoding="utf-8") as file:
    file.write("2025-02-28: Battery replaced again after a second low voltage warning.\n")
print("after a file changed:", load(conn, FOLDER))
print("documents:", conn.execute("SELECT COUNT(*) FROM documents").fetchone()[0])


first load, changed: 48 documents
second load, changed: []
after a file changed: ['Oslo/2025-02.txt']
documents: 48


The first load inserted every file, and the second found every document's text unchanged: the
`WHERE` on `DO UPDATE` skipped the write, so no row came back and nothing touched the index. After
the line was added, the third load rewrote exactly one document, still one row of 48. A path is
stored relative to the folder, so the archive does not depend on where the folder is. A load is one
transaction, so a file that fails to load leaves the archive as the last load left it.

### Searching the archive

`search` is the **Full-Text Search** notebook's search function, over documents instead of notes:
the words from a search box, an optional station, the best matches first, and a snippet of every
document:


In [5]:
def search(conn, text, station=None, limit=3):
    """The documents that best match the words in text, optionally at one station, with a snippet of each."""
    return conn.execute("""
        SELECT documents.path, trim(replace(snippet(documents_index, 0, '[', ']', '...', 10), char(10), ' ')) AS snippet
        FROM documents_index JOIN documents ON documents.id = documents_index.rowid
        WHERE documents_index MATCH ? AND (? IS NULL OR documents.station = ?)
        ORDER BY documents_index.rank, documents.id
        LIMIT ?
    """, (as_words(text), station, station, limit)).fetchall()


for row in search(conn, "logger failed"):
    print(row["path"], row["snippet"])
print()
for row in search(conn, "above freezing", station="Svalbard"):
    print(row["path"], row["snippet"])


Svalbard/2025-03.txt ...Data [logger] [failed] overnight, and there are no readings for...

Svalbard/2025-07.txt ...[Above] [freezing] all day, between 0.3 and 7.8...
Svalbard/2025-08.txt ...[Above] [freezing] all day, between 0.8 and 7.4...
Svalbard/2025-06.txt ...[Above] [freezing] all day, between 0.1 and 6.7...


Only one document in 48 holds both `logger` and `failed`: Svalbard's March, with the day its data
logger failed. The second search kept to Svalbard, which had days above freezing from morning to
night in only three months, and `bm25` ranked July first, where every day was. A snippet can run
across the end of a line, so `search` replaces the line breaks in it with spaces and trims the ends.
Every row answers to a column name, since `open_archive` set `sqlite3.Row` as the row factory, as
the **Row Factories** notebook did.

### A document whose file has gone

Files leave the folder as well as arriving in it. The office has taken Svalbard's March out of it,
and `load` does not notice: it adds and updates, and never deletes. Taking the document out of the
archive is a `DELETE`, and the delete trigger is what takes its words out of the index with it:


In [6]:
(FOLDER / "Svalbard" / "2025-03.txt").unlink()
print("the load after the file went:", load(conn, FOLDER))
print("the search still finds it:  ", [row["path"] for row in search(conn, "logger failed")])

with transaction(conn):
    removed = conn.execute("DELETE FROM documents WHERE path = ?", ("Svalbard/2025-03.txt",)).rowcount
print("rows deleted:", removed, "| documents left:", conn.execute("SELECT COUNT(*) FROM documents").fetchone()[0])
print("the search now finds:      ", [row["path"] for row in search(conn, "logger failed")])


the load after the file went: []
the search still finds it:   ['Svalbard/2025-03.txt']
rows deleted: 1 | documents left: 47
the search now finds:       []


The load wrote nothing and reported nothing, and the search went on finding a document whose file had
gone: a loader that only adds and updates leaves the archive holding what the folder has dropped. The
`DELETE` put that right, and the delete trigger handed FTS5 the old text, which it needs in order to
take those words out of an index that does not store the text itself. Sweeping up every file that has
gone, in one pass over the folder, is the third of the **Your turn** tasks.

### A lookup that uses an index

Not every question is a search. A page listing a station's summer asks for documents by station and
month, and `show_plan`, from the **Indexes and Query Plans** notebook, shows how SQLite finds them:


In [7]:
def show_plan(conn, sql, parameters=()):
    """Print the plan SQLite chooses for a statement, one step to a line, indented under the step it belongs to."""
    depth = {0: -1}
    for step, parent, _, detail in conn.execute("EXPLAIN QUERY PLAN " + sql, parameters):
        depth[step] = depth[parent] + 1
        print("    " + "  " * depth[step] + detail)


SUMMER = "SELECT path FROM documents WHERE station = ? AND month BETWEEN ? AND ? ORDER BY month"
show_plan(conn, SUMMER, ("Svalbard", "2025-06", "2025-08"))
print([row["path"] for row in conn.execute(SUMMER, ("Svalbard", "2025-06", "2025-08"))])
print("by month alone:")
show_plan(conn, "SELECT path FROM documents WHERE month = ?", ("2025-07",))


    SEARCH documents USING INDEX documents_by_station_month (station=? AND month>? AND month<?)
['Svalbard/2025-06.txt', 'Svalbard/2025-07.txt', 'Svalbard/2025-08.txt']
by month alone:
    SCAN documents


SQLite searches `documents_by_station_month` for the station and the range of months, and needs no
separate sort, since the index keeps a station's months in order. The same question asked by month
alone cannot use that index, because the index begins with the station, so SQLite scans every
document instead. A scan of 47 costs nothing, and a scan of years of them would.

### Version 2: a mark for reviewed documents

People have started reading the archive, and want to mark what they have read. That is a new column,
so it is a new migration, appended to the list. `migrate` runs it on this archive, which is at
version 1, and a second run finds nothing to do:


In [8]:
def add_reviewed(conn):
    """Version 2: a mark for the documents someone has reviewed, 0 until someone has."""
    conn.execute("ALTER TABLE documents ADD COLUMN reviewed INTEGER NOT NULL DEFAULT 0")


MIGRATIONS = [create_archive, add_reviewed]


print("applied:", migrate(conn, MIGRATIONS))
print("applied on a second run:", migrate(conn, MIGRATIONS))

SUMMER_MONTHS = ("Svalbard", "2025-06", "2025-08")
with transaction(conn):
    conn.execute("UPDATE documents SET reviewed = 1 WHERE station = ? AND month BETWEEN ? AND ?", SUMMER_MONTHS)

with (FOLDER / "Svalbard" / "2025-07.txt").open("a", encoding="utf-8") as file:
    file.write("2025-07-31: Fence repaired after a polar bear leaned on it.\n")
print("changed:", load(conn, FOLDER))
REVIEWS = "SELECT path, reviewed FROM documents WHERE station = ? AND month BETWEEN ? AND ? ORDER BY month"
print([tuple(row) for row in conn.execute(REVIEWS, SUMMER_MONTHS)])


applied: [2]
applied on a second run: []
changed: ['Svalbard/2025-07.txt']
[('Svalbard/2025-06.txt', 1), ('Svalbard/2025-07.txt', 1), ('Svalbard/2025-08.txt', 1)]


`ALTER TABLE ADD COLUMN` gave every document `reviewed = 0` without rewriting a row, and the
migration stamped version 2 in the same transaction. Svalbard's three summer months were marked, then
July's file changed and was loaded again, and all three marks remain, since the upsert sets only
`body`. Whether a changed document should lose its mark is the archive's decision to make: adding
`reviewed = 0` to the `SET` would do it. `open_archive` reads `MIGRATIONS` when it runs, so from now
on every archive it opens reaches version 2.

### Searching while the loader writes

A search page and the nightly loader use the archive at the same time. Here the loader has upserted
Tromso's December, with a new line about an aurora, and has not committed yet, while `reader`, a
second connection, searches:


In [9]:
reader = open_archive(ARCHIVE)
december = FOLDER / "Tromso" / "2025-12.txt"
with december.open("a", encoding="utf-8") as file:
    file.write("2025-12-31: Aurora seen over the station at midnight.\n")

with transaction(conn):
    conn.execute(UPSERT_DOCUMENT, ("Tromso/2025-12.txt", "Tromso", "2025-12", december.read_text(encoding="utf-8")))
    print("while the load is open, a search for aurora finds:", [row["path"] for row in search(reader, "aurora")])
print("after its commit:", [row["path"] for row in search(reader, "aurora")])
reader.close()


while the load is open, a search for aurora finds: []
after its commit: ['Tromso/2025-12.txt']


The search ran at once, during the loader's transaction, and saw the archive as the last commit left
it, without the aurora. After the commit, the same search found it. That is WAL mode, set by
`open_archive`, as the **Concurrency and WAL** notebook set it: in the rollback journal mode, a
search still reading its results would have kept the loader's commit waiting, which the Common
errors show.

### A checked backup, searched and downloaded

`back_up` is the **Backup and Copying** notebook's nightly backup, named after the archive:
`VACUUM INTO` a file whose name marks it unchecked, `integrity_check` on the copy, and a rename to
the night's name. The copy holds the full-text index too, so it searches as the archive does:


In [10]:
BACKUPS = SCRATCH / "backups"
BACKUPS.mkdir()

def back_up(archive, folder, night):
    """Copy an archive in use to folder/<name>-<night>.db, unless that night's backup exists, checking the copy first."""
    target = folder / f"{archive.stem}-{night}.db"
    if target.exists():
        return target
    unchecked = folder / f"{archive.stem}-{night}.unchecked"
    unchecked.unlink(missing_ok=True)
    source = sqlite3.connect(archive, autocommit=True)
    try:
        source.execute("VACUUM INTO ?", (str(unchecked),))
    finally:
        source.close()
    copy = sqlite3.connect(unchecked)
    try:
        integrity = copy.execute("PRAGMA integrity_check").fetchone()[0]
    finally:
        copy.close()
    if integrity != "ok":
        raise RuntimeError(f"the backup for {night} failed its check: {integrity}")
    return unchecked.rename(target)


backup = back_up(ARCHIVE, BACKUPS, "2026-01-01")
copy = sqlite3.connect(backup)
copy.row_factory = sqlite3.Row
print(backup.name, "| version:", copy.execute("PRAGMA user_version").fetchone()[0],
      "| documents:", copy.execute("SELECT COUNT(*) FROM documents").fetchone()[0])
print("a search of the backup for aurora finds:", [row["path"] for row in search(copy, "aurora")])
copy.close()

try:
    from google.colab import files
except ImportError:
    print("not running in Colab, so the backup stays at", backup)
else:
    files.download(str(backup))


archive-2026-01-01.db | version: 2 | documents: 47
a search of the backup for aurora finds: ['Tromso/2025-12.txt']
not running in Colab, so the backup stays at scratch/backups/archive-2026-01-01.db


The backup is at version 2, with every document, and found the aurora committed a moment before.
`back_up` opened a connection of its own, since `VACUUM INTO` needs a connection with no transaction
open. In Colab, the browser then saves the backup to your computer.

### Which way to load a document

| Write | When | Why |
|---|---|---|
| `INSERT` | a document that cannot already be there | a second run fails with `UNIQUE constraint failed`, which is right only when a second copy is a mistake |
| `INSERT ... ON CONFLICT (path) DO NOTHING` | documents that never change once written | a second run skips them, and a file that has changed is never loaded again |
| `INSERT ... ON CONFLICT (path) DO UPDATE SET body = excluded.body WHERE documents.body IS NOT excluded.body` | documents that change, in a table with columns the loader does not own | only a changed text is written, the row keeps its `id` and its other columns, and `RETURNING` names what changed |
| `INSERT OR REPLACE` | rows that nothing points at and nobody adds to | it deletes the old row and inserts a new one, with a new `id`, defaults in every other column, and no delete trigger |

The default for a loader is `DO UPDATE` with a `WHERE` and `RETURNING`, as `load` does.

### Three nights of the archive

The pieces of this notebook in one job. `last_month.db` is an archive made by last month's code, at
version 1, from the folder as it was in December, before the three lines this notebook added.
`nightly` opens the archive, which brings it up to date, loads the folder, and backs the archive up.
On the second night, a new month arrives:


In [11]:
LAST_MONTH = SCRATCH / "last_month.db"
write_logbook(SCRATCH / "logbook_in_december")
old = sqlite3.connect(LAST_MONTH, autocommit=True)
migrate(old, [create_archive])
load(old, SCRATCH / "logbook_in_december")
old.close()


def nightly(folder, archive, backups, night):
    """The night's whole job: open the archive, which brings it up to date, load the folder, and back the archive up."""
    conn = open_archive(archive)
    try:
        changed = load(conn, folder)
    finally:
        conn.close()
    return changed, back_up(archive, backups, night)


def describe(archive):
    """An archive's version and how many documents it holds."""
    conn = sqlite3.connect(archive)
    try:
        version = conn.execute("PRAGMA user_version").fetchone()[0]
        return f"version {version}, {conn.execute('SELECT COUNT(*) FROM documents').fetchone()[0]} documents"
    finally:
        conn.close()


print("before:", describe(LAST_MONTH))
for night in ("2026-01-01", "2026-01-02", "2026-01-03"):
    if night == "2026-01-02":
        (FOLDER / "Tromso" / "2026-01.txt").write_text(
            "Tromso, January 2026\n\n2026-01-01: Battery replaced after the first storm of the year.\n", encoding="utf-8")
    changed, backup = nightly(FOLDER, LAST_MONTH, BACKUPS, night)
    print(night, "changed:", changed, "| backup:", backup.name)
print("after: ", describe(LAST_MONTH))

copy = sqlite3.connect(backup)
copy.row_factory = sqlite3.Row
for row in search(copy, "battery replaced", station="Tromso"):
    print(row["path"], row["snippet"])
copy.close()


before: version 1, 48 documents
2026-01-01 changed: ['Oslo/2025-02.txt', 'Svalbard/2025-07.txt', 'Tromso/2025-12.txt'] | backup: last_month-2026-01-01.db
2026-01-02 changed: ['Tromso/2026-01.txt'] | backup: last_month-2026-01-02.db
2026-01-03 changed: [] | backup: last_month-2026-01-03.db
after:  version 2, 49 documents
Tromso/2026-01.txt ...[Battery] [replaced] after the first storm of the year.
Tromso/2025-01.txt ...[Battery] [replaced] after a low voltage warning. 2025-01-09...
Tromso/2025-05.txt ...[Battery] [replaced] after a low voltage warning. 2025-05-04...


The first night migrated last month's archive to version 2 before loading, and the load found the
three files that had changed since December. The second night loaded the new month, and the third
found nothing to do, and every night left a checked backup. The newest backup, searched for a battery
at Tromso, ranks the new January first, the shortest document with both words. Nothing in `nightly`
knew which version the archive was at, or which files had changed: the version in the file and the
upsert decided both.

### Where each part came from

| In the archive | What it relies on | The notebook that showed it |
|---|---|---|
| `STRICT`, and `UNIQUE` on `path` | a column that refuses the wrong type, and a path that cannot appear twice | **Type Affinity**, **Constraints** |
| `ON CONFLICT (path) DO UPDATE ... RETURNING id` | a loader that can run again, and says what it changed | **Constraints** |
| a `?` for every value, the file name in `VACUUM INTO` included | values that never become part of the SQL | **Parameters** |
| `sqlite3.Row` in `open_archive` | rows that answer to a column name | **Row Factories** |
| `transaction`, on a connection with `autocommit=True` | a load that happens completely or not at all | **Transactions**, **autocommit and isolation_level** |
| `documents_by_station_month`, and `show_plan` | a lookup that searches an index instead of scanning | **Indexes and Query Plans** |
| `documents_index`, with `content = 'documents'` and three triggers | an index that reads its text from the table and stays in step | **Full-Text Search** |
| `ORDER BY rank`, `snippet` and `as_words` | the best match first, the words around it, and a search box that cannot break the query | **Full-Text Search** |
| `MIGRATIONS`, `migrate` and `PRAGMA user_version` | an archive from last month's code, brought up to date | **Changing a Schema** |
| `PRAGMA journal_mode = WAL` in `open_archive` | a search that runs while the loader writes | **Concurrency and WAL** |
| `back_up`, with `VACUUM INTO` and a rename | a checked copy of an archive in use | **Backup and Copying** |


## Your turn

Six tasks. Write your answer in the cell under each task and run it.

Try a task before you look at its answer. Reading a solution teaches you much less than getting
there yourself, even slowly.

When you are ready: [**open the solutions notebook**](https://colab.research.google.com/github/johnfisher-ai/Python-Visual-Guides/blob/main/notebooks/sqlite3-deep-dive/19-a-searchable-archive-solutions.ipynb).

**1.** Search the archive for `ice on the anemometer` at Tromso, and print the path and snippet of
the three best documents.


In [12]:
# your code here


**2.** Count the documents at every station that mention cutting the grass, with one query that joins
the full-text index to `documents`.


In [13]:
# your code here


**3.** Write `remove_missing(conn, folder)`, which deletes the documents whose files are gone, in one
transaction, and returns their paths. Delete Svalbard's March file, and show that a search for
`logger failed` no longer finds it.


In [14]:
# your code here


**4.** Write version 3 of the archive, an index on `(station, reviewed)`, apply it with `migrate`,
and show the plan of a lookup for one station's documents that nobody has reviewed.


In [15]:
# your code here


**5.** Mark every document at Oslo reviewed, add a line to Oslo's May, and write a loader whose
upsert also sets `reviewed` back to 0 for a document whose text changed.


In [16]:
# your code here


**6.** Back up the archive with `conn.backup` into a database in memory, and search the copy for
`battery replaced`.


In [17]:
# your code here


## Common errors

### sqlite3.IntegrityError: UNIQUE constraint failed: documents.path


In [18]:
INSERT_DOCUMENT = "INSERT INTO documents (path, station, month, body) VALUES (?, ?, ?, ?)"


def load_with_insert(conn, folder):
    """A first loader: INSERT every file, in one transaction."""
    with transaction(conn):
        for path in sorted(folder.glob("*/*.txt")):
            text = path.read_text(encoding="utf-8")
            conn.execute(INSERT_DOCUMENT, (path.relative_to(folder).as_posix(), path.parent.name, path.stem, text))


insert_only = open_archive(SCRATCH / "insert_only.db")
load_with_insert(insert_only, FOLDER)
load_with_insert(insert_only, FOLDER)


IntegrityError: UNIQUE constraint failed: documents.path

The first night loaded every file. The second night's first `INSERT` found `Bergen/2025-01.txt`
already there, and the whole transaction rolled back, so a loader like this can run exactly once,
and no file changed afterwards would ever reach the archive. Upsert by the path instead, as `load`
does:


In [19]:
print("documents after the failed night:", insert_only.execute("SELECT COUNT(*) FROM documents").fetchone()[0])
print("changed by load:", load(insert_only, FOLDER))
insert_only.close()


documents after the failed night: 48
changed by load: []


### UnicodeDecodeError: 'utf-8' codec can't decode byte 0xf8 in position 79: invalid start byte


In [20]:
survey = FOLDER / "Kirkenes" / "2025-12.txt"
survey.parent.mkdir()
survey.write_bytes("Kirkenes, December 2025\n\n2025-12-01: Site surveyed for a new station near Tromsø.\n".encode("cp1252"))
load(conn, FOLDER)


UnicodeDecodeError: 'utf-8' codec can't decode byte 0xf8 in position 79: invalid start byte

The survey came from an older program that wrote Windows' cp1252, where `ø` is the single byte
`0xF8`, a byte UTF-8 never uses. `read_text` raised, and `transaction` rolled the whole load back, so
the archive is exactly as it was. A loader that has to take such files every night could decode the
bytes itself, falling back to cp1252. For one file, convert it to UTF-8 once, from the encoding it
was written in, and load again, which also picks up Tromso's January from the three nights:


In [21]:
print("documents after the failed load:", conn.execute("SELECT COUNT(*) FROM documents").fetchone()[0])
survey.write_text(survey.read_text(encoding="cp1252"), encoding="utf-8")
print("changed:", load(conn, FOLDER))
print([row["path"] for row in search(conn, "Tromsø")])


documents after the failed load: 47
changed: ['Kirkenes/2025-12.txt', 'Tromso/2026-01.txt']
['Kirkenes/2025-12.txt']


### sqlite3.OperationalError: no such column: reviewed


In [22]:
MADE_BY_VERSION_1 = SCRATCH / "made_by_version_1.db"
made = sqlite3.connect(MADE_BY_VERSION_1, autocommit=True)
migrate(made, [create_archive])
load(made, FOLDER)
made.close()

report = sqlite3.connect(MADE_BY_VERSION_1)
report.execute("SELECT path FROM documents WHERE station = ? AND reviewed = 0", ("Svalbard",)).fetchall()


OperationalError: no such column: reviewed

A report written for version 2 opened an archive that code at version 1 had made, and that nothing
had opened since, so the column it asked about does not exist there yet. Open an archive through
`open_archive`, which runs every migration the archive lacks before anything reads it:


In [23]:
report.close()
report = open_archive(MADE_BY_VERSION_1)
unreviewed = report.execute("SELECT path FROM documents WHERE station = ? AND reviewed = 0", ("Svalbard",)).fetchall()
print(len(unreviewed), "of Svalbard's documents not yet reviewed")
report.close()


11 of Svalbard's documents not yet reviewed


### sqlite3.OperationalError: duplicate column name: reviewed


In [24]:
add_reviewed(conn)


OperationalError: duplicate column name: reviewed

`add_reviewed` is version 2, and this archive has had it. A migration called by hand cannot know
that. `migrate` can, since it reads `user_version` first, and it runs a migration and stamps its
number in one transaction, so the two never disagree:


In [25]:
print("version:", conn.execute("PRAGMA user_version").fetchone()[0], "| applied:", migrate(conn, MIGRATIONS))


version: 2 | applied: []


### sqlite3.OperationalError: database is locked


In [26]:
NO_WAL = SCRATCH / "no_wal.db"
setup = sqlite3.connect(NO_WAL, autocommit=True)
migrate(setup, MIGRATIONS)
load(setup, FOLDER)
setup.close()
(FOLDER / "Bergen" / "2026-01.txt").write_text("Bergen, January 2026\n\n2026-01-02: Fence repaired after a storm.\n",
                                               encoding="utf-8")

page = sqlite3.connect(NO_WAL, autocommit=True, timeout=0.1)
results = page.execute("SELECT path FROM documents ORDER BY path")
print("the page shows:", results.fetchone()[0])
loader = sqlite3.connect(NO_WAL, autocommit=True, timeout=0.1)
load(loader, FOLDER)


the page shows: Bergen/2025-01.txt


OperationalError: database is locked

This archive was made without `open_archive`, so it is in the rollback journal mode. The page read
one row of its results and stopped, as a page showing the first few results does, and an unfinished
read keeps a shared lock. The loader's commit needs the file to itself, waited its tenth of a
second, and failed, and `transaction` rolled the load back. Close the page's results, then open both
connections through `open_archive`, which puts the archive in WAL mode, where a commit goes to the
`-wal` file and never waits for a reader:


In [27]:
results.close()
page.close()
loader.close()

page = open_archive(NO_WAL)
results = page.execute("SELECT path FROM documents ORDER BY path")
print("the page shows:", results.fetchone()["path"])
loader = open_archive(NO_WAL)
print("changed while the page reads:", load(loader, FOLDER))
results.close()
page.close()
loader.close()


the page shows: Bergen/2025-01.txt
changed while the page reads: ['Bergen/2026-01.txt']


### sqlite3.OperationalError: cannot VACUUM from within a transaction


In [28]:
def load_and_back_up(conn, folder, target):
    """Load the folder and take a backup in the same transaction, so that the two always match."""
    with transaction(conn):
        for path in sorted(folder.glob("*/*.txt")):
            name = path.relative_to(folder).as_posix()
            text = path.read_text(encoding="utf-8")
            conn.execute(UPSERT_DOCUMENT, (name, path.parent.name, path.stem, text)).fetchall()
        conn.execute("VACUUM INTO ?", (str(target),))


load_and_back_up(conn, FOLDER, BACKUPS / "matching.db")


OperationalError: cannot VACUUM from within a transaction

`VACUUM INTO` cannot run inside a transaction, the load's included, so the load rolled back with it.
The backup needs no transaction shared with the load to match it: taken after the commit, it copies
one committed state, which holds the whole load. Commit the load, then back up:


In [29]:
print("changed:", load(conn, FOLDER))
print("backup:", back_up(ARCHIVE, BACKUPS, "2026-01-04").name)


changed: ['Bergen/2026-01.txt']
backup: archive-2026-01-04.db


### No error, and a review mark gone: INSERT OR REPLACE in the loader


In [30]:
REPLACE_DOCUMENT = "INSERT OR REPLACE INTO documents (path, station, month, body) VALUES (?, ?, ?, ?)"
JULY = FOLDER / "Svalbard" / "2025-07.txt"


def state_of_july(conn):
    """Svalbard's July: its id and review mark, and how many index entries there are for a polar bear."""
    document = tuple(conn.execute("SELECT id, reviewed FROM documents WHERE path = 'Svalbard/2025-07.txt'").fetchone())
    matches = "SELECT COUNT(*) FROM documents_index WHERE documents_index MATCH ?"
    entries = conn.execute(matches, (as_words("polar bear"),)).fetchone()[0]
    return f"(id, reviewed) {document}, index entries for a polar bear: {entries}"


print("before:", state_of_july(conn))
with transaction(conn):
    conn.execute(REPLACE_DOCUMENT, ("Svalbard/2025-07.txt", "Svalbard", "2025-07", JULY.read_text(encoding="utf-8")))
print("after: ", state_of_july(conn))


before: (id, reviewed) (31, 1), index entries for a polar bear: 1
after:  (id, reviewed) (52, 0), index entries for a polar bear: 2


`INSERT OR REPLACE` found the path taken, so it deleted that row and inserted a new one: a new `id`,
and `reviewed` back to its default, so the mark is gone. SQLite also runs no delete trigger for a row
that `REPLACE` removes, unless `PRAGMA recursive_triggers` is on, so the index kept the old
document's words beside the new document's, and counts two documents about a polar bear where the
archive has one. Rebuild the index, which reads every document again, put the mark back, and load
with `load`, whose `DO UPDATE` changes only `body`:


In [31]:
with transaction(conn):
    conn.execute("INSERT INTO documents_index (documents_index) VALUES ('rebuild')")
    conn.execute("UPDATE documents SET reviewed = 1 WHERE path = 'Svalbard/2025-07.txt'")
print("after the rebuild:", state_of_july(conn))

with JULY.open("a", encoding="utf-8") as file:
    file.write("2025-07-31: The polar bear was gone by the evening.\n")
print("changed:", load(conn, FOLDER))
print("after the load:   ", state_of_july(conn))


after the rebuild: (id, reviewed) (52, 1), index entries for a polar bear: 1
changed: ['Svalbard/2025-07.txt']
after the load:    (id, reviewed) (52, 1), index entries for a polar bear: 1


Last, this cell closes `conn`, the last connection still open, and removes the scratch folder, with
the logbook, every archive and every backup in it:


In [32]:
conn.close()
shutil.rmtree("scratch")

print("scratch still there:", Path("scratch").exists())


scratch still there: False


## Recap

- An archive's loader upserts by path, `ON CONFLICT (path) DO UPDATE ... WHERE ... RETURNING`, so it
  can run every night, writes only what changed, keeps what people added, and says what it did.
- An external content FTS5 index, with triggers on the table, searches documents by their words and
  ranks them, and a snippet shows where the words are.
- `EXPLAIN QUERY PLAN` shows whether a lookup searches an index or scans, and an index on the columns
  of a lookup turns a scan into a search.
- Every change to the schema is a numbered migration, stamped in `user_version` in the same
  transaction, and opening an archive runs the migrations it lacks.
- WAL mode lets searches read while the loader writes, and a load is one transaction, so a failure
  leaves the archive as it was.
- A backup is taken after the load commits, with `VACUUM INTO`, checked before it gets its name, and
  searched and downloaded like the archive itself.


## What is next

The **Everyday Requests** notebook closes the guide with two more databases and the questions people
bring to whoever looks after them: a college's students, sections and grades, and a company's
invoices and payments. Every request in it is answered with what this guide has covered, joins,
aggregates, dates and updates that change exactly the rows they should.


---

&#8592; **Previous:** [Backup and Copying](https://colab.research.google.com/github/johnfisher-ai/Python-Visual-Guides/blob/main/notebooks/sqlite3-deep-dive/18-backup-and-copying.ipynb)  &nbsp;·&nbsp;  [sqlite3, Deep Dive Notebooks](https://johnfisher-ai.github.io/Python-Visual-Guides/sqlite3-deep-dive.html)  &nbsp;·&nbsp;  **Next:** [Everyday Requests](https://colab.research.google.com/github/johnfisher-ai/Python-Visual-Guides/blob/main/notebooks/sqlite3-deep-dive/20-everyday-requests.ipynb) &#8594;
